# Assignment 02: SlowFast 4x16 R50 on THETIS RGB (Google Colab)

Reproduction of Hovad et al. (2024), arXiv:2402.02545.

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** Allow Google Drive access when asked.
3. Keep this tab open. Training takes roughly 4–6 hours.

**If Colab disconnects:** reconnect and press **Runtime → Run all** again. Checkpoints are saved to Google Drive every epoch, so training continues from the last finished epoch.

In [ ]:
# 1. Google Drive: checkpoints and results live here so a disconnect loses nothing
from google.colab import drive
drive.mount('/content/drive')
RUN = '/content/drive/MyDrive/slowfast_thetis_run'
import os; os.makedirs(RUN, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code + dependencies
REPO_URL = 'https://github.com/<your-username>/slowfast-thetis-repro.git'   # <-- edit
%cd /content
!rm -rf slowfast-thetis-repro && git clone -q $REPO_URL
%cd /content/slowfast-thetis-repro
!pip install -q pytorchvideo==0.1.5 fvcore

In [ ]:
# 3. THETIS: download ONLY the RGB videos (about 3.7 GB, 1,980 clips) from the official GitHub repo
%cd /content
if not os.path.exists('/content/thetis/VIDEO_RGB/smash'):
    !git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/THETIS-dataset/dataset.git thetis
    !cd thetis && git sparse-checkout init --no-cone && git sparse-checkout set 'VIDEO_RGB/' && git checkout -q main
!find /content/thetis/VIDEO_RGB -name '*.avi' | wc -l
%cd /content/slowfast-thetis-repro

In [ ]:
# 4. Index videos, cache frames (short side 256), 70/20/10 split, dataset figures
CFG = 'configs/slowfast_4x16_r50_thetis.yaml'
OPTS = (f'data.root=/content/thetis/VIDEO_RGB data.cache_dir=/content/frames '
        f'data.split_csv={RUN}/splits.csv train.output_dir={RUN} data.num_workers=2')
!python src/prepare_thetis.py --config $CFG --opts $OPTS | head -5

In [ ]:
# 5. Model summary (shapes, 33.6 M params, GFLOPs)
!python src/model.py --config $CFG --out $RUN/model_summary.json | tail -3

In [ ]:
# 6. Train from scratch (196 epochs, resumes automatically from Drive)
# Out of GPU memory? add  train.batch_size=8  to OPTS in cell 4 and re-run from there.
!python src/train.py --config $CFG --opts $OPTS --time-limit-hours 11

In [ ]:
# 7. Test: 2 clips x 3 crops, metrics, confusion matrix, comparison table
!python src/evaluate.py --config $CFG --opts $OPTS

In [ ]:
# 8. Show results
from IPython.display import Image, Markdown, display
import json
display(Markdown(open(f'{RUN}/comparison_table.md').read()))
print(json.dumps(json.load(open(f'{RUN}/results.json')), indent=1))
for f in ['training_curves', 'confusion_matrix', 'per_class_accuracy', 'class_distribution', 'sample_frames']:
    display(Image(f'{RUN}/figures/{f}.png', width=900))

In [ ]:
# 9. One zip with everything for the report (checkpoints left out)
!cd /content/drive/MyDrive && zip -qr results_bundle.zip slowfast_thetis_run -x '*.pt' && ls -lh results_bundle.zip
print('Download results_bundle.zip from Google Drive (My Drive).')